In [1]:
# Setup

In [2]:
EPOCHS = 150
BATCH_SIZE = 32
VAL_SPLIT = 0.33

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [3]:
# Manifest

In [4]:
job_types = [0, 1, 2]     # Full-time, Part-time, Contract
settings = [0, 1, 2]      # Remote, Hybrid, On-site
exp_levels = [0, 1, 2]    # Junior, Mid, Senior

mapping = {
    "job_types": ["full-time", "part-time", "contract"],
    "settings": ["remote", "hybrid", "on-site"],
    "exp_levels": ["junior", "mid", "senior"]
}

In [5]:
# Training

In [6]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

X_train = np.random.randint(0,3,size=(5000,6))

# 1. Extract the Job and User columns for comparison
job_types = X_train[:, 0]
job_settings = X_train[:, 1]
job_exps = X_train[:, 2]

user_types = X_train[:, 3]
user_settings = X_train[:, 4]
user_exps = X_train[:, 5]

# 2. Calculate matches (True/False arrays) multiplied by your weights
type_scores = (job_types == user_types).astype(int) * 50
setting_scores = (job_settings == user_settings).astype(int) * 30
exp_scores = (job_exps == user_exps).astype(int) * 20

# 3. Sum them up to get total scores for each of the 5000 rows
total_scores = type_scores + setting_scores + exp_scores

# 4. Apply the threshold to create the binary labels (0 or 1)
y_train = (total_scores >= 60).astype(int)

In [7]:
# The brain

In [8]:
from tensorflow.keras.callbacks import EarlyStopping

model = tf.keras.Sequential([
    layers.Dense(12, activation='relu', input_shape=(6,)), # Hidden Layer
    layers.Dense(1, activation='sigmoid')                 # Output Layer
])

# Add the Hidden Layer (12 neurons, 6 inputs, ReLU). 2 neurons per input is standard.
model.add(Dense(12, input_shape=(6,), activation='relu'))

# Add the Output Layer (1 neuron, Sigmoid). We simply need "0" for disliked, "1" for liked. 1 neuron for output is enough.
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

model.save("mlp_model.keras")

early_stopping = EarlyStopping(monitor='val_loss', patience=3)

model.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=[early_stopping]
)

/home/bram/.local/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1774352120.402776  493503 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 30467 MB memory:  -> device: 0, name: Quadro RTX 8000, pci bus id: 0000:83:00.0, compute capability: 7.5
I0000 00:00:1774352120.403484  493503 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 11374 MB memory:  -> device: 1, name: NVIDIA TITAN X (Pascal), pci bus id: 0000:02:00.0, compute capability: 6.1
I0000 00:00:1774352120.404028  493503 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 11374 MB memory:  -> device: 2, name: NVIDIA TITAN X (Pascal), pci bus id:

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 12)             │            84 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            13 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 12)             │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            13 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 134 (536.00 B)

 Trainable params: 134 (536.00 B)

...

Epoch 1/150


I0000 00:00:1774352122.491969  493677 service.cc:152] XLA service 0x7f71d8003e40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774352122.491996  493677 service.cc:160]   StreamExecutor device (0): Quadro RTX 8000, Compute Capability 7.5
I0000 00:00:1774352122.492006  493677 service.cc:160]   StreamExecutor device (1): NVIDIA TITAN X (Pascal), Compute Capability 6.1
I0000 00:00:1774352122.492010  493677 service.cc:160]   StreamExecutor device (2): NVIDIA TITAN X (Pascal), Compute Capability 6.1
2026-03-24 12:35:22.580585: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1774352122.913045  493677 cuda_dnn.cc:529] Loaded cuDNN version 91002


 52/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2053 - loss: 0.7601

I0000 00:00:1774352123.719131  493677 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


105/105 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.5127 - loss: 0.6982 - val_accuracy: 0.8038 - val_loss: 0.6341
Epoch 2/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8205 - loss: 0.5859 - val_accuracy: 0.8038 - val_loss: 0.5524
Epoch 3/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8205 - loss: 0.5097 - val_accuracy: 0.8038 - val_loss: 0.5005
Epoch 4/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8205 - loss: 0.4727 - val_accuracy: 0.8038 - val_loss: 0.4894
Epoch 5/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8205 - loss: 0.4637 - val_accuracy: 0.8038 - val_loss: 0.4857
Epoch 6/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8205 - loss: 0.4581 - val_accuracy: 0.8038 - val_loss: 0.4782
Epoch 7/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8205 - loss: 0.4487 - val_accuracy: 0.8038 - val_loss: 0.4638
Epoch 8/150
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8205 - loss: 0.4274 - val_accuracy: 0.803

In [9]:
model.predict([[2,2,2,0,0,0]])

ValueError: Unrecognized data type: x=[[2, 2, 2, 0, 0, 0]] (of type <class 'list'>)

In [ ]:
model.predict([[0,0,0,0,1,2]])

In [ ]:
model.predict([[0,0,0,0,1,1]])

In [ ]:
model.predict([[0,1,0,0,2,0]])

In [ ]:
model.predict([[0,1,2,0,2,2]])

In [ ]:
model.predict([[0,0,0,0,0,0]])

In [10]:
tensorflowjs_converter \
    --input_format=tf_saved_model \
    --output_node_names='MobilenetV1/Predictions/Reshape_1' \
    --saved_model_tags=serve \
    /mobilenet/saved_model \
    /mobilenet/web_model

SyntaxError: cannot assign to expression (2103312714.py, line 1)